# End-to-end walkthrough

One synthetic session through every implemented package:

| stage | module | what this notebook shows |
|---|---|---|
| stream contract | `brainaccess_connection` | montage, simulated driver chunk, recording layout |
| cleaning | `preprocessing` | per-window notch -> band-pass -> resample -> z-score |
| windows | `classification.dataset` | labels at window start, boundary drops |
| dataset | `classification.dataset` | preprocessed windows plus the preprocessing snapshot |
| features | `features` | frozen REVE embeddings and cache metadata |
| training | `classification.training` | Optuna + grouped CV, final calibrated SVM |
| reporting | `reporting` | metrics, predictions, figures and `report.md` |
| packaging | `model_store` | `model.joblib`, `metadata.json`, `model_card.md` |
| inference | `inference` | validate the bundle and classify a raw window |

Live hardware streaming is covered by the
[`brainaccess_connection` notebooks](../brainaccess_connection/01_halo_connection.ipynb).
The last section loads the saved bundle with `hotb_2026.inference.Predictor`,
which rebuilds the recorded preprocessing and frozen REVE encoder. The
[inference notebook](../inference/01_live_inference.ipynb) covers `predict_raw`,
the `Prediction` contract and a simulated live stream in more detail.

Run this from the repository root with `uv run jupyter lab`, or execute it
headless with `nbclient` (see the last section).


In [1]:
from pathlib import Path
import json
import tempfile

import numpy as np
from IPython.display import Markdown, display

from hotb_2026.brainaccess_connection import MONTAGE, SimulatedDriver
from hotb_2026.preprocessing import PreprocessingConfig, Preprocessor
from hotb_2026.classification.dataset import (
    build_dataset,
    iter_windows,
    label_at,
    make_synthetic_recording,
)
from hotb_2026.classification.training import (
    TrainingConfig,
    embed_dataset,
    train_from_embeddings,
)
from hotb_2026.features import ReveEncoder
from hotb_2026.inference import Predictor
from hotb_2026.model_store import (
    StoreConfig,
    read_metadata,
    save_model,
)
from hotb_2026.reporting import write_run

np.set_printoptions(precision=3, suppress=True)
workdir = Path(tempfile.mkdtemp(prefix="hotb-walkthrough-"))
print("workdir:", workdir)


workdir: /var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-walkthrough-4jfe5yan


/Users/dominikmika/PycharmProjects/HOTB_2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Stream contract - `brainaccess_connection`

Every downstream package consumes the same layout: ordered electrode names and a
`(channels, samples)` signal with one timestamp per sample. Labels are a sorted
list of state-change timestamps; the label at a time is the state that is active
then.

`SimulatedDriver` produces the same chunks as the headset without Bluetooth or
the BrainAccess SDK.


In [2]:
print("Montage:", MONTAGE.names)
print("Hardware sample rate:", MONTAGE.sfreq, "Hz")
print("Bias electrode:", MONTAGE.bias_electrode)

driver = SimulatedDriver(MONTAGE, chunk_ms=40.0, seed=0)
driver.open()
driver.start()
chunk = driver.read()
driver.stop()
driver.close()

print("\nSimulated chunk:", chunk.shape, "(samples, channels)", chunk.dtype)
print("RMS per channel (uV):", np.round(np.sqrt((chunk ** 2).mean(axis=0)), 1))

recording = make_synthetic_recording(
    duration_s=180.0,
    sfreq=MONTAGE.sfreq,
    state_duration_s=30.0,
    seed=7,
    session_id="walkthrough-1",
)
print("\nRecording:", recording.session_id, recording.signal.shape)
print("State changes:", [(change.time_s, change.label) for change in recording.state_changes])
print("Current time:", recording.time_s[0], "...", recording.time_s[-1], "s")


Montage: ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
Hardware sample rate: 250.0 Hz
Bias electrode: 7

Simulated chunk: (10, 8) (samples, channels) float32
RMS per channel (uV): [17.1 17.4 17.8 15.3 14.7 15.  16.9 19. ]

Recording: walkthrough-1 (8, 45000)
State changes: [(0.0, 0), (30.0, 1), (60.0, 2), (90.0, 0), (120.0, 1), (150.0, 2)]
Current time: 0.0 ... 179.996 s


## 2. Preprocessing - `preprocessing`

A window is cleaned before it is encoded: notch -> band-pass -> resample
250 -> 200 Hz -> per-channel z-score -> clip. The `PreprocessingConfig` is the
single source of truth and is recorded in the dataset and later in the model
bundle.


In [3]:
preprocessing = PreprocessingConfig(sfreq=recording.sfreq)
preprocessor = Preprocessor(preprocessing)

raw_window = recording.signal[:, : int(10 * recording.sfreq)]
clean_window = preprocessor.process(raw_window)

print("config:", preprocessing)
print("in :", raw_window.shape, "-> out:", clean_window.shape, "at", preprocessor.output_sfreq, "Hz")
print(
    "RMS in/out:",
    round(float(np.sqrt(np.mean(raw_window ** 2))), 2),
    "/",
    round(float(np.sqrt(np.mean(clean_window ** 2))), 2),
)
print("channel means after z-score:", np.round(clean_window.mean(axis=1), 3))
print("range after clip:", round(float(clean_window.min()), 2), "...", round(float(clean_window.max()), 2))


config: PreprocessingConfig(sfreq=250.0, notch_freq=50.0, notch_q=30.0, bandpass=(0.5, 45.0), filter_order=4, target_sfreq=200.0, zscore=True, clip=15.0)
in : (8, 2500) -> out: (8, 2000) at 200.0 Hz
RMS in/out: 8.99 / 1.0
channel means after z-score: [-0.  0.  0.  0.  0.  0.  0. -0.]
range after clip: -5.29 ... 3.67


## 3. Windows and labels - `classification.dataset`

Windows are planned on the raw time axis. A window takes the label that is active
at its **start**; windows that cross a state change are dropped by default so the
label is unambiguous.


In [4]:
windows, dropped = iter_windows(recording, window_s=10.0)

print("planned windows:", len(windows))
print("dropped:", dropped)
for window in windows[:4]:
    print(
        f"start={window.start_s:5.1f}s end={window.end_s:5.1f}s "
        f"label={window.label} change={window.state_change_s:5.1f}s"
    )

print("label at 29 s:", label_at(recording, 29.0))
print("label at 30 s:", label_at(recording, 30.0))


planned windows: 18
dropped: {'unlabeled': 0, 'boundary': 0, 'short_state': 0, 'too_short': 0}
start=  0.0s end= 10.0s label=0 change=  0.0s
start= 10.0s end= 20.0s label=0 change=  0.0s
start= 20.0s end= 30.0s label=0 change=  0.0s
start= 30.0s end= 40.0s label=1 change= 30.0s
label at 29 s: 0
label at 30 s: 1


## 4. Dataset - `classification.dataset`

`build_dataset` cuts every recording, preprocesses each window independently and
records the exact `PreprocessingConfig`. With at least five sessions,
`group_by="auto"` keeps whole sessions intact during cross-validation.


In [5]:
recordings = [
    make_synthetic_recording(
        duration_s=180.0,
        sfreq=recording.sfreq,
        state_duration_s=30.0,
        seed=seed,
        session_id=f"walkthrough-{seed + 1}",
    )
    for seed in range(6)
]
dataset = build_dataset(
    recordings,
    window_s=10.0,
    preprocess_config=preprocessing,
    group_by="auto",
)
summary = dataset.summary()

print(
    json.dumps(
        {
            key: summary[key]
            for key in (
                "n_windows",
                "n_channels",
                "n_samples",
                "sfreq",
                "raw_sfreq",
                "window_s",
                "class_counts",
                "n_groups",
            )
        },
        indent=2,
    )
)
print("preprocessing snapshot:", dataset.preprocessing)


{
  "n_windows": 108,
  "n_channels": 8,
  "n_samples": 2000,
  "sfreq": 200.0,
  "raw_sfreq": 250.0,
  "window_s": 10.0,
  "class_counts": {
    "low": 36,
    "medium": 36,
    "high": 36
  },
  "n_groups": 6
}
preprocessing snapshot: {'sfreq': 250.0, 'notch_freq': 50.0, 'notch_q': 30.0, 'bandpass': (0.5, 45.0), 'filter_order': 4, 'target_sfreq': 200.0, 'zscore': True, 'clip': 15.0}


## 5. Features - `features`

The frozen REVE encoder turns each window into a 512-d embedding.
`embed_dataset` also writes a content-addressed cache, so repeating the walkthrough
reuses the embeddings instead of re-running the transformer.


In [6]:
encoder = ReveEncoder(dataset.electrodes)
cache_dir = workdir / "embeddings"
embeddings, embedding_meta = embed_dataset(
    dataset, encoder=encoder, cache_dir=str(cache_dir)
)

print("embeddings:", embeddings.shape, embeddings.dtype)
print("encoder:", embedding_meta["model_name"], "+", embedding_meta["pos_model_name"])
print("embedding dim / patch:", embedding_meta["embedding_dim"], "/", embedding_meta["patch_size"])
print("cache files:", sorted(path.name for path in cache_dir.iterdir()))


W1115 09:10:58.478000 30879 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 1/1 [00:00<00:00, 4609.13it/s]

flash_attn not found, install it with `pip install flash_attn` if you want to use it


Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 140/140 [00:00<00:00, 9779.86it/s]

embeddings: (108, 512) float32
encoder: brain-bzh/reve-base + brain-bzh/reve-positions
embedding dim / patch: 512 / 200
cache files: ['embeddings.json', 'embeddings.npy']


## 6. Training - `classification.training`

`train_from_embeddings` keeps the frozen embeddings and runs the Optuna search
with grouped cross-validation, then refits the selected pipeline on all windows.
`train` is the convenience wrapper that also builds the encoder and cache.


In [7]:
training_config = TrainingConfig(n_splits=3, n_trials=5, seed=0)
result = train_from_embeddings(
    embeddings,
    dataset.y,
    dataset.groups,
    config=training_config,
    label_names=dataset.label_names,
    dataset_summary=dataset.summary(),
    embedding_meta=embedding_meta,
)

print("best params:", result["best_params"])
print(
    "CV balanced accuracy:",
    round(result["cv"]["mean"], 3),
    "+-",
    round(result["cv"]["std"], 3),
)
print(
    "holdout balanced accuracy:",
    round(result["holdout"]["metrics"]["balanced_accuracy"], 3),
)
print("final model:", type(result["model"]).__name__)
print("classes:", result["model"].classes_.tolist())


best params: {'C': 1.0, 'gamma': 0.001953125, 'class_weight': 'balanced'}
CV balanced accuracy: 0.843 +- 0.035
holdout balanced accuracy: 1.0
final model: Pipeline
classes: [0, 1, 2]


## 7. Reporting - `reporting`

`write_run` persists the configuration, metrics, trials and predictions, renders
figures and writes a Markdown `report.md`.


In [8]:
run_dir = workdir / "run"
write_run(run_dir, result, make_plots=True)

for path in sorted(run_dir.iterdir()):
    print(f"{path.name:24} {path.stat().st_size / 1024:7.1f} KiB")

display(Markdown((run_dir / "report.md").read_text()[:1500]))


best_params.json             0.4 KiB
confidence.png              20.9 KiB
config.json                  0.2 KiB
confusion_matrix.png        23.8 KiB
dataset_summary.json         0.7 KiB
embedding_pca.png           45.3 KiB
embeddings.meta.json         0.7 KiB
metrics.json                 3.4 KiB
predictions.npz              2.1 KiB
report.md                    1.9 KiB
trials.csv                   0.6 KiB
trials.png                  26.8 KiB


# REVE + SVM training report

Generated: 2026-11-15T09:11:00

## Run

- Metric: `balanced_accuracy`
- Window: 10.000 s at 200.000 Hz
- Trials: 5 (best CV score 0.843)
- Groups: 5 train / 1 held out

## Dataset

- Windows: 108
- Channels: 8
- Samples per window: 2000
- Raw rate: 250.000 Hz
- Class counts: low: 36, medium: 36, high: 36

## Embeddings

- Encoder: `brain-bzh/reve-base`
- Shape: [108, 512]
- Cache: `/var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-walkthrough-4jfe5yan/embeddings`

## Hyperparameters

| parameter | value |
|---|---|
| `C` | 1.00000 |
| `gamma` | 0.00195 |
| `class_weight` | balanced |

Cross-validation was used to select these values, so its score is optimistic. The held-out score below is the honest estimate.

## Held-out test groups

- Windows: 18
- Groups: walkthrough-1
- Accuracy: 1.000
- Balanced accuracy: 1.000
- Macro F1: 1.000
- ROC AUC (OVR): 1.000
- Mean confidence: 0.919 (correct 0.919, wrong —)

### Confusion matrix

| true \ pred | low | medium | high |
|---|---|---|---|
| **low** | 6 | 0 | 0 |
| **medium** | 0 | 6 | 0 |
| **high** | 0 | 0 | 6 |

### Per class

| class | precision | recall | f1 | support |
|---|---|---|---|---|
| low | 1.000 | 1.000 | 1.000 | 6 |
| medium | 1.000 | 1.000 | 1.000 | 6 |
| high | 1.000 | 1.000 | 1.000 | 6 |

## Cross-validation (training groups)

- Folds: 3
- Scores: 0.889, 0.833, 0.806
- Mean: 0.843 ± 0.035

## Out-of-fold predictions

- Accuracy: 0.844
- Balanced accuracy: 0.844
- Macro F1: 0.843



## 8. Bundle - `model_store`

`save_model` validates the fitted pipeline, then writes `model.joblib`, a
self-describing `metadata.json` and a Hugging Face `model_card.md`. Training
artifacts from `run_dir` are copied alongside. Publishing to the Hub is optional
and covered in `classification/02_save_and_publish.ipynb`.


In [9]:
bundle = save_model(
    result,
    workdir / "bundle",
    config=StoreConfig(version="0.1.0"),
    run_dir=run_dir,
)
print("bundle:", bundle.directory)
for path in bundle.files:
    print(f"{path.name:20} {path.stat().st_size / 1024:7.1f} KiB")

metadata = read_metadata(bundle.directory)
contract = {
    key: metadata[key]
    for key in (
        "model_version",
        "labels",
        "input",
        "encoder",
        "preprocessing",
    )
}
display(Markdown("```json\n" + json.dumps(contract, indent=2) + "\n```"))


bundle: /var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-walkthrough-4jfe5yan/bundle
metadata.json            3.7 KiB
metrics.json             3.4 KiB
model.joblib           239.3 KiB
model_card.md            1.7 KiB
predictions.npz          2.1 KiB
report.md                1.9 KiB


```json
{
  "model_version": "0.1.0",
  "labels": {
    "classes": [
      0,
      1,
      2
    ],
    "n_classes": 3,
    "names": [
      "low",
      "medium",
      "high"
    ]
  },
  "input": {
    "electrodes": [
      "F3",
      "F4",
      "C3",
      "C4",
      "P3",
      "P4",
      "O1",
      "O2"
    ],
    "label_rule": "state active at window start",
    "n_samples": 2000,
    "raw_sfreq": 250.0,
    "sfreq": 200.0,
    "time_units": "s",
    "window_s": 10.0
  },
  "encoder": {
    "embedding_dim": 512,
    "model_name": "brain-bzh/reve-base",
    "patch_size": 200,
    "pos_model_name": "brain-bzh/reve-positions",
    "revision": null
  },
  "preprocessing": {
    "bandpass": [
      0.5,
      45.0
    ],
    "clip": 15.0,
    "filter_order": 4,
    "notch_freq": 50.0,
    "notch_q": 30.0,
    "sfreq": 250.0,
    "target_sfreq": 200.0,
    "zscore": true
  }
}
```

## 9. Live inference - `inference`

The bundle is a joblib pipeline plus `metadata.json`. `Predictor.from_bundle`
validates the metadata against the serialized model, rebuilds the recorded
preprocessing and the frozen REVE encoder, and classifies raw windows. The
returned `Prediction` carries the label, probabilities, current time, the time
the current predicted state started, the window bounds and the model version.

`state_change_time` is not a recorded ground-truth change: live inference only
knows when the *predicted* label changed. A real deployment can pin the bundle
with `Predictor.from_pretrained(..., revision="0.1.0")`.


In [10]:
predictor = Predictor.from_bundle(bundle.directory)
prediction = predictor.predict_raw(
    recording.signal[:, : predictor.raw_samples],
    current_time=predictor.window_s,
)

label_names = metadata["labels"]["names"]
probabilities = [prediction.probabilities[name] for name in label_names]

print(predictor)
print(
    f"window {prediction.window_start:g}-{prediction.window_end:g}s -> "
    f"class {prediction.label} ({prediction.label_name}) "
    f"p={prediction.confidence:.3f}"
)
print("probabilities:", np.round(probabilities, 3).tolist())
print("predicted state since:", prediction.state_change_time, "s")
print("model version:", prediction.model_version)


Predictor(model_version='0.1.0', window_s=10, electrodes=['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2'])
window 0-10s -> class 0 (low) p=0.993
probabilities: [0.993, 0.007, 0.0]
predicted state since: 10.0 s
model version: 0.1.0


## 10. Validate everything

```bash
# Full test suite (dataset, training, reporting, model_store, streamer, ...).
uv run pytest -q

# Inspect a saved bundle without loading it in Python.
uv run python -m hotb_2026.model_store show bundles/nback
```

Deeper dives:

- [`preprocessing/01_preprocessing.ipynb`](../preprocessing/01_preprocessing.ipynb) - filter and resampling details.
- [`reve_model/01_reve_embddings.ipynb`](../reve_model/01_reve_embddings.ipynb) - REVE encoder internals.
- [`classification/01_training.ipynb`](../classification/01_training.ipynb) - full training run with plots.
- [`classification/02_save_and_publish.ipynb`](../classification/02_save_and_publish.ipynb) - bundle and Hugging Face upload.
- [`brainaccess_connection/01_halo_connection.ipynb`](../brainaccess_connection/01_halo_connection.ipynb) - real headset streaming.
